# Semantic Search Engine Interactive Activity
*AIPI 510: Data Sourcing for Analytics*

*Week 13*



---



**Build Your Own Semantic Search Engine**

> **Goal**: Create a semantic search app from
scratch and deploy it to Hugging Face Spaces.


1.   Pick a dataset
2.  Chunk + embed your data
3. Store your data in a Chroma vector db
4. Build a search app using Streamlit or Gradio
5. Publish on Hugging Face Spaces
6. Submit Link on Canvas



---

*This is a starter notebook. This is designed to give you a jumping off point only. *

Make sure to
`pip install chromadb gradio sentence-transformers`


In [1]:
# Load an embedding model (you can use any embedding model)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

c:\Users\colby\aipi\sourcing_data\semantic_search_engine\Semantic-Search-Engine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\colby\aipi\sourcing_data\semantic_search_engine\Semantic-Search-Engine\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\colby\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks o

In [3]:
import kagglehub
import os
import pandas as pd

path = kagglehub.dataset_download("priyamchoksi/pokemon-cards")
file = [f for f in os.listdir(path) if f.endswith((".csv", ".parquet"))][0]
full_path = os.path.join(path, file)

df = pd.read_csv(full_path) if file.endswith(".csv") else pd.read_parquet(full_path)

texts = (df["name"].fillna("") + " " + df["caption"].fillna("")).str.strip().tolist()

In [ ]:
# Generate embeddings
embeddings = model.encode(texts)

In [ ]:
# Store embeddings in ChromaDB
import chromadb
from chromadb.config import Settings
client = chromadb.Client(Settings(anonymized_telemetry=False))
collection = client.create_collection("my_collection")
collection.add(documents=texts, embeddings=embeddings.tolist(), ids=[str(i) for i in range(len(texts))])

In [ ]:
# Define a search function
def semantic_search(query):
    query_embedding = model.encode([query])
    results = collection.query(query_embeddings=query_embedding.tolist(), n_results=1)
    return "\n\n".join(results['documents'][0])

In [ ]:
# Simple Gradio interface
import gradio as gr
demo = gr.Interface(
    fn=semantic_search,
    inputs=gr.Textbox(label="Enter your search query"),
    outputs=gr.Textbox(label="Top Matches"),
    title="Semantic Search Engine",
    description="Search over your custom dataset using semantic similarity."
)

demo.launch()

### Next Steps:

- Save your chromadb
- Write code to load your chromadb
- Convert your loading, search function and app code to an app.py script
- Create a requirements.txt file
- Test your code!!
- Push your git repo to Hugging Face Spaces


___
Your file tree in your repository will look like this:

- chromadb/
  - chroma.sqlite3
- README.md
- requirements.txt
- app.py